In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")
    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. gold_fato_diario

In [0]:
gold_fato_diario  = ler_ultima_particao_tabela_spark(spark,  "workspace.case_spark_cvm.gold_fato_diario")

In [0]:
display(gold_fato_diario)

In [0]:
window_ultima_data = Window.partitionBy("cnpj_fundo_classe")

gold_fato_diario = gold_fato_diario\
    .withColumn(
        "ultima_dt_fundo",
        f.max(f.col("dt_comptc")).over(window_ultima_data)
)


In [0]:
gold_cubo_rentabilidade = gold_fato_diario\
    .filter(f.col("dt_comptc") == f.col("ultima_dt_fundo"))\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        "retorno_diario",
        "retorno_21d",
        "retorno_63d",
        "retorno_126d",
        "retorno_252d",
        "retorno_inicio",
        "re"
    )

In [0]:
display(gold_fato_diario)

In [0]:
gold_cubo_rentabilidade